***Downloading dataset***

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('otto-group-product-classification-challenge')

print("Path to competition files:", path)

In [1]:
from pathlib import Path
import pandas as pd

def load_housing_data(x):
    return pd.read_csv(x, low_memory=False)

train= load_housing_data("/kaggle/input/competitions/otto-group-product-classification-challenge/train.csv")
test= load_housing_data("/kaggle/input/competitions/otto-group-product-classification-challenge/test.csv")

In [2]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61878 entries, 0 to 61877
Data columns (total 95 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   id       61878 non-null  int64 
 1   feat_1   61878 non-null  int64 
 2   feat_2   61878 non-null  int64 
 3   feat_3   61878 non-null  int64 
 4   feat_4   61878 non-null  int64 
 5   feat_5   61878 non-null  int64 
 6   feat_6   61878 non-null  int64 
 7   feat_7   61878 non-null  int64 
 8   feat_8   61878 non-null  int64 
 9   feat_9   61878 non-null  int64 
 10  feat_10  61878 non-null  int64 
 11  feat_11  61878 non-null  int64 
 12  feat_12  61878 non-null  int64 
 13  feat_13  61878 non-null  int64 
 14  feat_14  61878 non-null  int64 
 15  feat_15  61878 non-null  int64 
 16  feat_16  61878 non-null  int64 
 17  feat_17  61878 non-null  int64 
 18  feat_18  61878 non-null  int64 
 19  feat_19  61878 non-null  int64 
 20  feat_20  61878 non-null  int64 
 21  feat_21  61878 non-null  int64 
 22

In [5]:
train["target"].value_counts

<bound method IndexOpsMixin.value_counts of 0        Class_1
1        Class_1
2        Class_1
3        Class_1
4        Class_1
          ...   
61873    Class_9
61874    Class_9
61875    Class_9
61876    Class_9
61877    Class_9
Name: target, Length: 61878, dtype: object>

In [6]:
train[train.isna().any(axis=1)]

,id,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,feat_9,...,feat_85,feat_86,feat_87,feat_88,feat_89,feat_90,feat_91,feat_92,feat_93,target


In [9]:
train.columns[train.isna().any()]

Index([], dtype='object')

In [2]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.base import clone
from sklearn.dummy import DummyClassifier

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

# ------------------------------------------------------------------
# 1. Target: object -> integer labels
# ------------------------------------------------------------------
drop_cols    = [c for c in ["id","target"] if c in train.columns]
feature_cols = [c for c in train.columns if c not in drop_cols]

X = train[feature_cols].copy()

le = LabelEncoder()
y = pd.Series(le.fit_transform(train["target"]),
              index=train.index, name="target")          # "Class_1".."Class_9" -> 0..8
CLASSES = list(le.classes_)
N_CLASSES = len(CLASSES)
print(f"classes ({N_CLASSES}): {CLASSES}")
print("class counts:\n", pd.Series(y).value_counts().sort_index())

# target must NOT be a feature
assert "target" not in feature_cols

num_cols = X.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
print(f"num_cols={len(num_cols)} cat_cols={len(cat_cols)}")

# ------------------------------------------------------------------
# 2. Shared preprocessing (LGBM + XGB need numeric input)
# ------------------------------------------------------------------
pre = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median"))]), num_cols),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OrdinalEncoder(handle_unknown="use_encoded_value",
                                  unknown_value=-1)),
    ]), cat_cols),
])

# ------------------------------------------------------------------
# 3. Multi-class classifiers
# ------------------------------------------------------------------
lgbm_clf = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=N_CLASSES,
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=31,
    min_child_samples=40,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
    device_type="gpu",
)

xgb_clf = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=N_CLASSES,
    n_estimators=3000,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    tree_method="hist",          # "gpu_hist" if GPU
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

cat_clf = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=3.0,
    loss_function="MultiClass",
    eval_metric="MultiClass",
    random_seed=42,
    verbose=False,
    task_type="GPU",
)

models = {
    "lightgbm": Pipeline([("pre", pre), ("clf", lgbm_clf)]),
    "xgboost":  Pipeline([("pre", pre), ("clf", xgb_clf)]),
    "catboost": cat_clf,
}

# ------------------------------------------------------------------
# 4. Stratified 10-fold CV with multi-class log loss
# ------------------------------------------------------------------
cat_idx = [feature_cols.index(c) for c in cat_cols]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(skf.split(X, y))          # stratify on y

results = {n: {"train": [], "val": [], "oof": np.zeros((len(y), N_CLASSES))}
           for n in models}
base_scores = []

for fold, (tr_idx, val_idx) in enumerate(folds, start=1):
    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    # prior baseline: predicts class frequencies
    base = DummyClassifier(strategy="prior").fit(X_tr, y_tr)
    bs = log_loss(y_val, base.predict_proba(X_val), labels=np.arange(N_CLASSES))
    base_scores.append(bs)

    line = f"Fold {fold:2d} | baseline={bs:.4f}"
    for name, est in models.items():
        m = clone(est)
        if name == "catboost":
            m.fit(X_tr, y_tr, cat_features=cat_idx)
        else:
            m.fit(X_tr, y_tr)

        tr_prob = m.predict_proba(X_tr)
        vl_prob = m.predict_proba(X_val)

        tr_ll = log_loss(y_tr, tr_prob, labels=np.arange(N_CLASSES))
        vl_ll = log_loss(y_val, vl_prob, labels=np.arange(N_CLASSES))
        results[name]["train"].append(tr_ll)
        results[name]["val"].append(vl_ll)
        results[name]["oof"][val_idx] = vl_prob
        line += f" | {name}={vl_ll:.4f}"
    print(line)

# ------------------------------------------------------------------
# 5. Summary (lower log loss = better)
# ------------------------------------------------------------------
def pooled_logloss(name):
    return log_loss(y, results[name]["oof"], labels=np.arange(N_CLASSES))

summary = pd.DataFrame([{
    "model": n,
    "mean_train_logloss": np.mean(results[n]["train"]),
    "mean_val_logloss":   np.mean(results[n]["val"]),
    "std_val_logloss":    np.std(results[n]["val"]),
    "pooled_oof_logloss": pooled_logloss(n),
    "train_val_gap":      np.mean(results[n]["train"]) - np.mean(results[n]["val"]),
} for n in models]).sort_values("pooled_oof_logloss")

print("\n=== Multi-class log loss comparison (lower is better) ===")
print(summary.to_string(index=False, float_format=lambda v: f"{v:.5f}"))
print(f"\nBaseline (prior) mean val log loss: {np.mean(base_scores):.5f}")

# ------------------------------------------------------------------
# 6. Blends (probabilities must be averaged, then re-normalised)
# ------------------------------------------------------------------
def blend_probs(names, weights=None):
    probs = np.mean([results[n]["oof"] for n in names], axis=0) \
            if weights is None else \
            np.tensordot(weights, np.stack([results[n]["oof"] for n in names]),
                         axes=1)
    probs = probs / probs.sum(axis=1, keepdims=True)
    return probs

blend_all = blend_probs(list(models))
print(f"Equal-weight blend (all 3) log loss: "
      f"{log_loss(y, blend_all, labels=np.arange(N_CLASSES)):.5f}")

blend_2 = blend_probs(["xgboost", "lightgbm"])
print(f"XGB+LGBM blend log loss:             "
      f"{log_loss(y, blend_2, labels=np.arange(N_CLASSES)):.5f}")

# ------------------------------------------------------------------
# 7. Submission helper (Otto wants probabilities per class)
# ------------------------------------------------------------------
# test = pd.read_csv("test.csv")
# test_prob = np.mean([...fold models...], axis=0)   # average fold predictions
# sub = pd.DataFrame(test_prob, columns=CLASSES)
# sub.insert(0, "id", test["id"].values)
# sub.to_csv("submission.csv", index=False)

classes (9): ['Class_1', 'Class_2', 'Class_3', 'Class_4', 'Class_5', 'Class_6', 'Class_7', 'Class_8', 'Class_9']
class counts:
 target
0     1929
1    16122
2     8004
3     2691
4     2739
5    14135
6     2839
7     8464
8     4955
Name: count, dtype: int64
num_cols=93 cat_cols=0


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature 

Fold  1 | baseline=1.9504 | lightgbm=0.5103 | xgboost=0.4675 | catboost=0.5115


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold  2 | baseline=1.9505 | lightgbm=0.5219 | xgboost=0.4718 | catboost=0.5116


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


... (2 outputs left out)

Fold  4 | baseline=1.9505 | lightgbm=0.4942 | xgboost=0.4561 | catboost=0.5024


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold  5 | baseline=1.9502 | lightgbm=0.5079 | xgboost=0.4624 | catboost=0.5062

=== Multi-class log loss comparison (lower is better) ===
   model  mean_train_logloss  mean_val_logloss  std_val_logloss  pooled_oof_logloss  train_val_gap
 xgboost             0.16188           0.46197          0.00719             0.46197       -0.30009
catboost             0.41005           0.50501          0.00679             0.50501       -0.09496
lightgbm             0.05903           0.50676          0.00954             0.50676       -0.44773

Baseline (prior) mean val log loss: 1.95041
Equal-weight blend (all 3) log loss: 0.45420
XGB+LGBM blend log loss:             0.46669


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
